# 🏎️ Semana 2 — API Jolpica-F1 + Dataset Maestro
**Módulos del curso:** Módulo 7 (API) + Módulo 8 (Data Wrangling I)

> ⚠️ **Nota:** La API original de Ergast fue dada de baja en 2024. Su reemplazo oficial
> es **Jolpica-F1**, que tiene exactamente la misma estructura de endpoints.
> Solo cambia la URL base.

### Objetivos de esta semana
1. Consumir la **API Jolpica-F1** para traer datos actualizados de la temporada 2024
2. Unir todas las tablas en un único **dataset maestro** (`f1_master.csv`)
3. Calcular features derivadas: `positions_gained`, `top3`, `finished`
4. Exportar el dataset listo para el EDA de la semana siguiente


In [1]:
import pandas as pd
import numpy as np
import requests
import time
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas')

✅ Librerías cargadas


---
## Parte 1 — Cargar el dataset local


In [2]:
DATA = '../data'

circuits     = pd.read_csv(f'{DATA}/circuits.csv')
constructors = pd.read_csv(f'{DATA}/constructors.csv')
drivers      = pd.read_csv(f'{DATA}/drivers.csv')
races        = pd.read_csv(f'{DATA}/races.csv')
results      = pd.read_csv(f'{DATA}/results.csv')
qualifying   = pd.read_csv(f'{DATA}/qualifying.csv')
pit_stops    = pd.read_csv(f'{DATA}/pit_stops.csv')
standings    = pd.read_csv(f'{DATA}/driver_standings.csv')
con_stand    = pd.read_csv(f'{DATA}/constructor_standings.csv')
status       = pd.read_csv(f'{DATA}/status.csv')

tablas = [('circuits',circuits),('constructors',constructors),
          ('drivers',drivers),('races',races),('results',results),
          ('qualifying',qualifying),('pit_stops',pit_stops)]

for name, df in tablas:
    print(f'  {name:<25} {df.shape[0]:>5} filas × {df.shape[1]:>2} cols')

print('\n✅ Todas las tablas cargadas')

  circuits                     20 filas ×  9 cols
  constructors                 13 filas ×  5 cols
  drivers                      20 filas ×  9 cols
  races                       166 filas ×  8 cols
  results                   26759 filas × 18 cols
  qualifying                  952 filas ×  9 cols
  pit_stops                   613 filas ×  7 cols

✅ Todas las tablas cargadas


---
## Parte 2 — API Jolpica-F1 (Módulo 7 del curso)

**Jolpica-F1** es el reemplazo oficial de la API de Ergast.
Misma estructura, mismos endpoints, solo cambia la URL base:

| | URL |
|---|---|
| Ergast (deprecada) | `https://ergast.com/api/f1/` |
| **Jolpica (actual)** | `https://api.jolpi.ca/ergast/f1/` |

> 📌 Documentación: https://github.com/jolpica/jolpica-f1


In [3]:
# URL base de la nueva API
BASE_URL = 'https://api.jolpi.ca/ergast/f1'

def get_f1_api(endpoint, limit=100, offset=0):
    """
    Consume un endpoint de la API Jolpica-F1.
    Retorna el JSON o None si hay error.
    """
    url = f'{BASE_URL}/{endpoint}.json?limit={limit}&offset={offset}'
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'⚠️  Error: {e}')
        return None

# Probar con el calendario 2024
data = get_f1_api('2024')
if data:
    races_api = data['MRData']['RaceTable']['Races']
    print(f'✅ API Jolpica-F1 responde correctamente')
    print(f'   Carreras en 2024: {len(races_api)}')
    print(f'   Primera carrera: {races_api[0]["raceName"]} — {races_api[0]["date"]}')
    print(f'   Última carrera:  {races_api[-1]["raceName"]} — {races_api[-1]["date"]}')
else:
    print('⚠️  Sin conexión a la API — usando solo datos locales')

✅ API Jolpica-F1 responde correctamente
   Carreras en 2024: 24
   Primera carrera: Bahrain Grand Prix — 2024-03-02
   Última carrera:  Abu Dhabi Grand Prix — 2024-12-08


In [4]:
def get_season_results(year):
    """
    Descarga todos los resultados de una temporada desde Jolpica-F1.
    Maneja la paginación automáticamente (la API devuelve 30 por defecto).
    """
    all_rows = []
    offset = 0
    limit = 100

    while True:
        data = get_f1_api(f'{year}/results', limit=limit, offset=offset)
        if not data:
            break

        races_data = data['MRData']['RaceTable']['Races']
        total = int(data['MRData']['total'])

        for race in races_data:
            for res in race.get('Results', []):
                all_rows.append({
                    'year':        year,
                    'round':       int(race['round']),
                    'race_name':   race['raceName'],
                    'circuit':     race['Circuit']['circuitName'],
                    'country':     race['Circuit']['Location']['country'],
                    'date':        race['date'],
                    'driver':      f"{res['Driver']['givenName']} {res['Driver']['familyName']}",
                    'driver_code': res['Driver'].get('code', ''),
                    'team':        res['Constructor']['name'],
                    'grid':        int(res.get('grid', 0)),
                    'position':    int(res['position']),
                    'points':      float(res.get('points', 0)),
                    'laps':        int(res.get('laps', 0)),
                    'status':      res.get('status', ''),
                })

        offset += limit
        if offset >= total:
            break
        time.sleep(0.5)  # respetar rate limit de la API

    return pd.DataFrame(all_rows)

# Descargar temporada 2024
print('Descargando temporada 2024...')
df_2024 = get_season_results(2024)

if not df_2024.empty:
    print(f'\n✅ Temporada 2024 descargada correctamente')
    print(f'   Resultados: {len(df_2024)} filas')
    print(f'   Carreras:   {df_2024["round"].nunique()}')
    print(f'   Ganadores:\n')
    ganadores = df_2024[df_2024['position']==1][['round','race_name','driver','team']]
    print(ganadores.to_string(index=False))
else:
    print('⚠️  No se pudo descargar 2024 — continuamos con datos locales')

Descargando temporada 2024...

✅ Temporada 2024 descargada correctamente
   Resultados: 479 filas
   Carreras:   24
   Ganadores:

 round                 race_name          driver     team
     1        Bahrain Grand Prix  Max Verstappen Red Bull
     2  Saudi Arabian Grand Prix  Max Verstappen Red Bull
     3     Australian Grand Prix    Carlos Sainz  Ferrari
     4       Japanese Grand Prix  Max Verstappen Red Bull
     5        Chinese Grand Prix  Max Verstappen Red Bull
     6          Miami Grand Prix    Lando Norris  McLaren
     7 Emilia Romagna Grand Prix  Max Verstappen Red Bull
     8         Monaco Grand Prix Charles Leclerc  Ferrari
     9       Canadian Grand Prix  Max Verstappen Red Bull
    10        Spanish Grand Prix  Max Verstappen Red Bull
    11       Austrian Grand Prix  George Russell Mercedes
    12        British Grand Prix  Lewis Hamilton Mercedes
    13      Hungarian Grand Prix   Oscar Piastri  McLaren
    14        Belgian Grand Prix  Lewis Hamilton Mercedes

In [5]:
# Veamos también cuántas victorias tuvo cada piloto en 2024
if not df_2024.empty:
    wins_2024 = (df_2024[df_2024['position']==1]
                 .groupby(['driver','team'])
                 .size()
                 .reset_index(name='victorias')
                 .sort_values('victorias', ascending=False))
    print('🏆 Victorias por piloto en 2024:')
    print(wins_2024.to_string(index=False))

🏆 Victorias por piloto en 2024:
         driver     team  victorias
 Max Verstappen Red Bull          9
   Lando Norris  McLaren          4
Charles Leclerc  Ferrari          3
 George Russell Mercedes          2
   Carlos Sainz  Ferrari          2
 Lewis Hamilton Mercedes          2
  Oscar Piastri  McLaren          2


---
## Parte 3 — Construir el Dataset Maestro

Unimos todas las tablas locales con `results` como tabla central.
Este es el **Data Wrangling** del Módulo 8.


In [6]:
# Convertir columnas numéricas
for col in ['position', 'grid', 'points']:
    results[col] = pd.to_numeric(results[col], errors='coerce')

# Preparar tablas auxiliares
races_slim = races[['raceId','year','round','circuitId','name']].rename(
    columns={'name':'race_name'})

drivers_slim = drivers[['driverId','code','forename','surname','nationality','dob']].copy()
drivers_slim['driver_name'] = drivers_slim['forename'] + ' ' + drivers_slim['surname']

constr_slim = constructors[['constructorId','name','nationality']].rename(
    columns={'name':'team', 'nationality':'team_nationality'})

circuits_slim = circuits[['circuitId','name','country','lat','lng']].rename(
    columns={'name':'circuit_name'})

status_slim = status[['statusId','status']]

# JOIN de todas las tablas
master = (results
    .merge(races_slim,    on='raceId',        how='left')
    .merge(drivers_slim,  on='driverId',       how='left')
    .merge(constr_slim,   on='constructorId',  how='left')
    .merge(circuits_slim, on='circuitId',      how='left')
    .merge(status_slim,   on='statusId',       how='left')
)

print(f'Dataset maestro: {master.shape[0]} filas × {master.shape[1]} columnas')
master.head(3)

Dataset maestro: 26759 filas × 35 columnas


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,nationality,dob,driver_name,team,team_nationality,circuit_name,country,lat,lng,status
0,1,18,1,1,22,1,1.0,1,1,10.0,...,British,1985-01-07,Lewis Hamilton,McLaren,British,NaN,NaN,NaN,NaN,Finished
1,2,18,2,2,3,5,2.0,2,2,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Finished
2,3,18,3,3,7,7,3.0,3,3,6.0,...,German,1985-06-27,Nico Rosberg,Williams,British,NaN,NaN,NaN,NaN,Finished


---
## Parte 4 — Features derivadas


In [7]:
# 1. Posiciones ganadas/perdidas
master['positions_gained'] = master['grid'] - master['position']

# 2. Variable objetivo binaria: ¿terminó en el podio?
master['top3'] = (master['position'] <= 3).astype(int)

# 3. ¿Terminó la carrera?
master['finished'] = (master['status'] == 'Finished').astype(int)

print('✅ Features derivadas creadas:')
print(f"  positions_gained — media: {master['positions_gained'].mean():.2f}")
print(f"  top3             — {master['top3'].sum()} resultados en podio ({master['top3'].mean()*100:.1f}%)")
print(f"  finished         — {master['finished'].sum()} pilotos terminaron ({master['finished'].mean()*100:.1f}%)")

✅ Features derivadas creadas:
  positions_gained — media: 2.96
  top3             — 3396 resultados en podio (12.7%)
  finished         — 7674 pilotos terminaron (28.7%)


In [8]:
# Agregar features de pit stops
pit_stops['duration'] = pd.to_numeric(pit_stops['duration'], errors='coerce')

pit_count = (pit_stops.groupby(['raceId','driverId'])['stop']
             .max().reset_index().rename(columns={'stop':'num_stops'}))

pit_time  = (pit_stops.groupby(['raceId','driverId'])['duration']
             .sum().reset_index().rename(columns={'duration':'total_pit_time'}))

pit_fast  = (pit_stops.groupby(['raceId','driverId'])['duration']
             .min().reset_index().rename(columns={'duration':'fastest_pit'}))

master = (master
    .merge(pit_count, on=['raceId','driverId'], how='left')
    .merge(pit_time,  on=['raceId','driverId'], how='left')
    .merge(pit_fast,  on=['raceId','driverId'], how='left')
)
master['num_stops'] = master['num_stops'].fillna(0).astype(int)

print(f'✅ Features de pit stops agregadas')
print(f'   Dataset final: {master.shape[0]} filas × {master.shape[1]} columnas')
print(f'   Columnas: {list(master.columns)}')

✅ Features de pit stops agregadas
   Dataset final: 26759 filas × 41 columnas
   Columnas: ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'race_name', 'code', 'forename', 'surname', 'nationality', 'dob', 'driver_name', 'team', 'team_nationality', 'circuit_name', 'country', 'lat', 'lng', 'status', 'positions_gained', 'top3', 'finished', 'num_stops', 'total_pit_time', 'fastest_pit']


In [9]:
# Resumen de nulos
print('📊 Valores nulos (columnas con nulos > 0):')
nulls = master.isnull().sum()
pct   = (nulls / len(master) * 100).round(1)
null_df = pd.DataFrame({'nulos': nulls, '%': pct})
print(null_df[null_df['nulos'] > 0].sort_values('nulos', ascending=False).to_string())

📊 Valores nulos (columnas con nulos > 0):
                  nulos     %
fastest_pit       26631  99.5
total_pit_time    26631  99.5
round             23212  86.7
circuitId         23212  86.7
year              23212  86.7
lat               23212  86.7
race_name         23212  86.7
country           23212  86.7
circuit_name      23212  86.7
lng               23212  86.7
nationality       23165  86.6
code              23165  86.6
forename          23165  86.6
driver_name       23165  86.6
surname           23165  86.6
dob               23165  86.6
team_nationality  16399  61.3
team              16399  61.3
position          10953  40.9
positions_gained  10953  40.9
status             6589  24.6


In [10]:
# Diagnóstico: ¿grid == position en el master antes de guardar?
igual = (master['grid'] == master['position']).sum()
total = len(master)
print(f"Filas donde grid == position: {igual} de {total} ({igual/total*100:.1f}%)")
print(f"\nEjemplos donde grid != position:")
diff = master[master['grid'] != master['position']][['race_name','driver_name','grid','position']].head(10)
print(diff.to_string())

Filas donde grid == position: 1863 de 26759 (7.0%)

Ejemplos donde grid != position:
   race_name      driver_name  grid  position
1        NaN              NaN     5       2.0
2        NaN     Nico Rosberg     7       3.0
3        NaN  Fernando Alonso    11       4.0
4        NaN              NaN     3       5.0
5        NaN              NaN    13       6.0
6        NaN              NaN    17       7.0
7        NaN   Kimi Räikkönen    15       8.0
8        NaN              NaN     2       NaN
9        NaN              NaN    18       NaN
10       NaN              NaN    19       NaN


In [11]:
# Exportar el dataset maestro
output_path = '../data/f1_master.csv'
master.to_csv(output_path, index=False)
import os
size_kb = round(os.path.getsize(output_path) / 1024, 1)
print('✅ f1_master.csv guardado en /data')
print(f'   Tamaño: {size_kb} KB')
print(f'   Shape:  {master.shape[0]} filas × {master.shape[1]} columnas')

✅ f1_master.csv guardado en /data
   Tamaño: 3132.4 KB
   Shape:  26759 filas × 41 columnas


---
## ✅ Resumen Semana 2

| Tarea | Estado |
|-------|--------|
| API Jolpica-F1 consumida (reemplaza Ergast) | ✅ |
| Temporada 2024 descargada desde la API | ✅ |
| Join de 6 tablas → dataset maestro | ✅ |
| Feature `positions_gained` = grid − position | ✅ |
| Feature `top3` (variable objetivo binaria) | ✅ |
| Features pit stops: `num_stops`, `total_pit_time`, `fastest_pit` | ✅ |
| `f1_master.csv` exportado a `/data` | ✅ |

### Próximos pasos — Semana 3
- Tratar nulos en `total_pit_time`, `fastest_pit` y columnas de tiempos
- Detectar y tratar outliers con IQR
- Normalizar formatos inconsistentes
- Notebook `03_limpieza.ipynb`


In [12]:
igual = (master['grid'] == master['position']).sum()
total = len(master)
print("grid == position:", igual, "de", total)
diff = master[master['grid'] != master['position']]
print("Filas con diferencia:", len(diff))
print(diff[['race_name','driver_name','grid','position']].head(10))

grid == position: 1863 de 26759
Filas con diferencia: 24896
   race_name      driver_name  grid  position
1        NaN              NaN     5       2.0
2        NaN     Nico Rosberg     7       3.0
3        NaN  Fernando Alonso    11       4.0
4        NaN              NaN     3       5.0
5        NaN              NaN    13       6.0
6        NaN              NaN    17       7.0
7        NaN   Kimi Räikkönen    15       8.0
8        NaN              NaN     2       NaN
9        NaN              NaN    18       NaN
10       NaN              NaN    19       NaN
